In [ ]:
!pip install pyspark pandas pyarrow -q

from google.colab import drive
drive.mount("/content/drive")

import os
import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("HM_Labeling_Downsampling")
    .config("spark.driver.memory", "8g")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "2g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

print("Spark da khoi tao xong.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark da khoi tao xong.


In [ ]:
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"

INPUT_TRANS = BASE_PATH + "processed_v2/cleaned_transactions.parquet"
MASTER_CAND_DIR = BASE_PATH + "outputs_v2/master/"
LABELED_DIR = BASE_PATH + "outputs_v2/labeled/"
FEATURE_DIR = BASE_PATH + "outputs_v2/features/"

os.makedirs(LABELED_DIR, exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

TRAIN_MASTER_PATH = MASTER_CAND_DIR + "train_master_candidates.parquet"
TEST_MASTER_PATH = MASTER_CAND_DIR + "test_master_candidates.parquet"

TRAIN_LABELED_PATH = LABELED_DIR + "train_labeled_sampled_10_1.parquet"
TEST_BASE_PATH = LABELED_DIR + "test_base_candidates.parquet"

print("Da khai bao xong duong dan.")

Da khai bao xong duong dan.


In [ ]:
transactions = spark.read.parquet(INPUT_TRANS)
train_master = spark.read.parquet(TRAIN_MASTER_PATH)
test_master = spark.read.parquet(TEST_MASTER_PATH)

print("Train master:", train_master.count())
print("Test master :", test_master.count())
print("Transactions:", transactions.count())

Train master: 54097747
Test master : 53853851
Transactions: 31788324


In [ ]:
max_date = transactions.select(F.max("t_dat_date")).collect()[0][0]

test_start = max_date - datetime.timedelta(days=7)
val_start = test_start - datetime.timedelta(days=7)

print("Max date  :", max_date)
print("Val start :", val_start)
print("Test start:", test_start)

Max date  : 2020-09-22
Val start : 2020-09-08
Test start: 2020-09-15


In [ ]:
target_w7 = (
    transactions
    .filter((F.col("t_dat_date") >= val_start) & (F.col("t_dat_date") < test_start))
    .select("customer_id", "article_id")
    .dropDuplicates()
    .withColumn("label", F.lit(1))
)

print("So cap positive trong tuan 7:", target_w7.count())

So cap positive trong tuan 7: 237152


In [ ]:
train_labeled = (
    train_master
    .join(F.broadcast(target_w7), ["customer_id", "article_id"], "left")
    .fillna({"label": 0})
)

label_stats = train_labeled.groupBy("label").count().orderBy("label")
label_stats.show()

print("Tong train sau gan nhan:", train_labeled.count())

+-----+--------+
|label|   count|
+-----+--------+
|    0|54076847|
|    1|   20900|
+-----+--------+

Tong train sau gan nhan: 54097747


In [ ]:
NEGATIVE_RATIO = 10
SEED = 42

positives = train_labeled.filter(F.col("label") == 1)
negatives = train_labeled.filter(F.col("label") == 0)

pos_count = positives.count()
neg_count = negatives.count()

neg_fraction = min(
    1.0,
    (pos_count * NEGATIVE_RATIO) / neg_count if neg_count > 0 else 1.0
)

negatives_sampled = negatives.sample(
    withReplacement=False,
    fraction=neg_fraction,
    seed=SEED
)

train_labeled_sampled = positives.unionByName(negatives_sampled)

print("Positive:", pos_count)
print("Negative:", neg_count)
print("Negative fraction:", neg_fraction)
print("Train sau downsampling:", train_labeled_sampled.count())

Positive: 20900
Negative: 54076847
Negative fraction: 0.0038648703020721605
Train sau downsampling: 230601


In [ ]:
train_labeled_sampled.write.mode("overwrite").parquet(TRAIN_LABELED_PATH)
test_master.write.mode("overwrite").parquet(TEST_BASE_PATH)

print("Da luu train labeled:", TRAIN_LABELED_PATH)
print("Da luu test base    :", TEST_BASE_PATH)

Da luu train labeled: /content/drive/MyDrive/HM-DATA/outputs_v2/labeled/train_labeled_sampled_10_1.parquet
Da luu test base    : /content/drive/MyDrive/HM-DATA/outputs_v2/labeled/test_base_candidates.parquet
